# 프로젝트 1 - Weekend 3: 메모리와 프롬프트 엔지니어링

| 항목 | 내용 |
|------|------|
| **프로젝트** | 주택청약 FAQ 챗봇 - 최종 완성 |
| **핵심 기술** | Few-shot, CoT, Memory, Gradio |
| **작성자** | 김민아 |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w3_memory_prompt_engineering_project/p1_weekend3_memory_and_deploy_260328_%E1%84%80%E1%85%B5%E1%86%B7%E1%84%86%E1%85%B5%E1%86%AB%E1%84%8B%E1%85%A1.ipynb)

## 환경 설정

In [ ]:
# Colab 환경: 필요한 패키지 설치
!pip install -q openai langchain-openai langchain-community langchain faiss-cpu python-dotenv gradio

In [ ]:
# ── Colab 환경 설정 ──────────────────────────────────────────────
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# ── 라이브러리 임포트 ────────────────────────────────────────────
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS

# ── 클라이언트 초기화 ────────────────────────────────────────────
client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ 환경 설정 완료")

## 📦 실습용 샘플 데이터

In [ ]:
# ============================================================
# 주택청약 FAQ 샘플 데이터 (실습용)
# ============================================================
SAMPLE_FAQ_DATA = [
    {"id": "FAQ001", "category": "청약통장",
     "question": "주택청약종합저축이란 무엇인가요?",
     "answer": "주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨",
     "keywords": ["청약종합저축", "만능통장", "납입", "가입"], "difficulty": "easy"},
    {"id": "FAQ004", "category": "청약통장",
     "question": "청약통장 1순위 조건은 무엇인가요?",
     "answer": "1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입",
     "keywords": ["1순위", "가입기간", "예치금", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ005", "category": "청약자격",
     "question": "주택 청약 신청 자격 조건은 무엇인가요?",
     "answer": "1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능",
     "keywords": ["청약자격", "만19세", "무주택", "세대주"], "difficulty": "easy"},
    {"id": "FAQ006", "category": "청약자격",
     "question": "무주택자 기준은 무엇인가요?",
     "answer": "본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함",
     "keywords": ["무주택", "세대원", "소형주택", "분양권"], "difficulty": "medium"},
    {"id": "FAQ009", "category": "특별공급",
     "question": "특별공급의 종류에는 어떤 것이 있나요?",
     "answer": "1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년 이내)\n4) 생애최초 (최초 주택 구입)\n5) 노부모부양 (만 65세 이상 부모)\n※ 2021년부터 신혼/생애최초 물량 확대",
     "keywords": ["특별공급", "기관추천", "다자녀", "신혼부부", "생애최초"], "difficulty": "medium"},
    {"id": "FAQ010", "category": "특별공급",
     "question": "신혼부부 특별공급 조건은 무엇인가요?",
     "answer": "1) 혼인기간 7년 이내 무주택 세대주\n2) 소득: 도시근로자 월평균소득 100~140%\n3) 전용면적 85㎡ 이하\n4) 혼인기간 짧을수록 + 자녀 많을수록 가점 높음\n5) 예비 신혼부부도 신청 가능",
     "keywords": ["신혼부부", "혼인기간", "소득기준", "가점"], "difficulty": "medium"},
    {"id": "FAQ013", "category": "일반공급",
     "question": "가점제와 추첨제의 차이는 무엇인가요?",
     "answer": "가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%",
     "keywords": ["가점제", "추첨제", "84점", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ017", "category": "당첨/계약",
     "question": "당첨자 발표는 어떻게 확인하나요?",
     "answer": "1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인",
     "keywords": ["당첨자발표", "청약홈", "SMS알림", "서류제출"], "difficulty": "easy"},
    {"id": "FAQ020", "category": "당첨/계약",
     "question": "재당첨 제한이란 무엇인가요?",
     "answer": "당첨 후 일정 기간 다른 주택 청약 불가:\n1) 투기과열지구: 10년\n2) 청약과열지역: 7년\n3) 수도권 공공주택: 5년\n※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)",
     "keywords": ["재당첨제한", "10년", "7년", "세대원"], "difficulty": "medium"},
    {"id": "FAQ023", "category": "기타",
     "question": "청약홈 사이트는 어떻게 이용하나요?",
     "answer": "청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공",
     "keywords": ["청약홈", "공인인증서", "간편인증", "가점계산"], "difficulty": "easy"},
]

SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ017"},
]

print(f"📦 FAQ 데이터 로드 완료: {len(SAMPLE_FAQ_DATA)}개 QA, {len(SAMPLE_TEST_QUERIES)}개 테스트 질의")

## 🔧 Weekend 2 복원

In [ ]:
# ── Weekend 2 복원: 벡터 스토어 + RAG 체인 ────────────────────────
# Weekend 2에서 만든 핵심 컴포넌트를 빠르게 복원합니다.
# (Document 변환 → FAISS 벡터 스토어 → Retriever → RAG 체인)

# FAQ → Document 변환
documents = [Document(
    page_content=f"질문: {f['question']}\n답변: {f['answer']}",
    metadata={"id": f["id"], "category": f["category"]}
) for f in SAMPLE_FAQ_DATA]

# FAISS 벡터 스토어 + Retriever
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Document 리스트 → 문자열 변환 함수
def format_docs(docs):
    return "\n---\n".join([f"[{d.metadata.get('category','')}] {d.page_content}" for d in docs])

# 기본 RAG 프롬프트 + 체인
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "주택청약 전문 상담원입니다. 참고 FAQ:\n{context}\n\nFAQ 기반으로 친절하게 답변하세요."),
    ("user", "{question}")
])

baseline_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print(f"✅ RAG 파이프라인 복원 완료 ({len(documents)}개 문서)")
print(f"테스트: {baseline_chain.invoke('청약통장이 뭐예요?')[:80]}...")

---
## 사이클 1: Weekend 2 복원 + 기준선 측정

Weekend 2의 RAG 체인을 복원하고, `SAMPLE_TEST_QUERIES` 5개로 응답 시간과 답변 길이의 기준선을 측정하세요.

In [ ]:
# ── 사이클 1: Weekend 2 복원 + 기준선 측정 ────────────────────────
# ⭐ 기준선(Baseline) 측정이란?
#   개선 전 성능을 먼저 기록해두어야
#   나중에 Few-shot, CoT, Memory 등을 적용했을 때
#   "얼마나 좋아졌는지" 비교할 수 있습니다.
#
# 측정 항목:
#   - 응답 시간(초): LLM API 호출 + 검색 시간
#   - 답변 길이(자): 답변의 상세도 파악
import time

results = []
for tq in SAMPLE_TEST_QUERIES:
    start = time.time()
    answer = baseline_chain.invoke(tq["query"])
    elapsed = round(time.time() - start, 2)
    results.append({"query": tq["query"], "time": elapsed, "length": len(answer)})
    print(f"❓ {tq['query']} → {elapsed}초, {len(answer)}자")

avg_time = sum(r["time"] for r in results) / len(results)
avg_len = sum(r["length"] for r in results) / len(results)
print(f"\n📊 기준선: 평균 {avg_time:.1f}초, 평균 {avg_len:.0f}자")

---
## 사이클 2: Few-shot 프롬프트

FAQ 답변 예시 3개를 포함한 few-shot 프롬프트를 만들고, 기본 프롬프트 대비 답변 형식이 개선되는지 비교하세요.

In [ ]:
# ── 사이클 2: Few-shot 프롬프트 ───────────────────────────────────
# ⭐ Few-shot Prompting이란?
#   LLM에게 "이런 식으로 답변해" 라는 예시를 몇 개 보여주는 기법
#
#   Zero-shot: 예시 없이 바로 질문 → 답변 형식이 들쭉날쭉
#   Few-shot:  예시 2~5개 제공   → LLM이 패턴을 학습하여 일관된 형식으로 답변
#
# 구현 방법:
#   ChatPromptTemplate에 (user, assistant) 쌍으로 예시를 넣음
#   → LLM은 이 예시의 형식(마크다운, 이모지 등)을 따라함
#
# 수업에서 배운 내용 (260324~260326):
#   - 예시는 3개 정도가 적당 (너무 많으면 토큰 낭비, 너무 적으면 효과 부족)
#   - 예시의 형식이 곧 출력 형식이 됨 → 원하는 형식을 예시에 정확히 반영!

fewshot_prompt = ChatPromptTemplate.from_messages([
    ("system", "주택청약 전문 상담원입니다. 참고 FAQ:\n{context}\n\n아래 예시처럼 답변하세요."),

    # 예시 1: 청약통장 (구조화된 마크다운 형식)
    ("user", "청약통장이 뭔가요?"),
    ("assistant",
     "## 📋 청약통장 안내\n\n"
     "**청약종합저축**은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n\n"
     "### 핵심 정보\n"
     "1. 매월 2만~50만원 자유 납입\n"
     "2. 가입 후 일정 기간 경과 시 청약 자격 부여\n\n"
     "📞 추가 문의: 청약홈(1644-7445)"),

    # 예시 2: 특별공급 (번호 리스트 형식)
    ("user", "특별공급 종류 알려주세요"),
    ("assistant",
     "## 📋 특별공급 종류 안내\n\n"
     "특별공급은 **5가지 유형**이 있습니다.\n\n"
     "### 종류\n"
     "1. 기관추천 (국가유공자, 장애인)\n"
     "2. 다자녀가구 (3명 이상)\n"
     "3. 신혼부부 (혼인 7년 이내)\n"
     "4. 생애최초 (최초 주택 구입)\n"
     "5. 노부모부양 (만 65세 이상)\n\n"
     "📞 추가 문의: 청약홈(1644-7445)"),

    # 예시 3: 당첨 확인 (절차 안내 형식)
    ("user", "당첨 확인은 어떻게?"),
    ("assistant",
     "## 📋 당첨 확인 방법\n\n"
     "### 확인 절차\n"
     "1. 청약홈(www.applyhome.co.kr) 접속\n"
     "2. 당첨자 조회 메뉴 클릭\n"
     "3. 문자 알림 서비스 신청 가능\n\n"
     "### ⚠️ 주의\n"
     "- 서류 제출 기간과 계약 일정 반드시 확인\n\n"
     "📞 추가 문의: 청약홈(1644-7445)"),

    # 실제 사용자 질문
    ("user", "{question}")
])

fewshot_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | fewshot_prompt | llm | StrOutputParser()
)

# ── 기본 프롬프트 vs Few-shot 프롬프트 비교 ──────────────────────
q = "가점제가 뭐예요?"
print("📋 기본 프롬프트 (형식 지정 없음):")
print(baseline_chain.invoke(q)[:200])
print("\n" + "=" * 60)
print("\n📋 Few-shot 프롬프트 (마크다운 + 이모지 형식 학습):")
print(fewshot_chain.invoke(q)[:200])

---
## 사이클 3: Chain-of-Thought 프롬프트

"1단계-문제 파악, 2단계-원인 분석, 3단계-해결 방법" 사고 과정을 명시하는 CoT 프롬프트를 만들고, 복잡한 질문 3개로 테스트하세요.

In [ ]:
# ── 사이클 3: Chain-of-Thought 프롬프트 ───────────────────────────
# ⭐ CoT (Chain-of-Thought) 란?
#   LLM에게 "단계별로 생각하라"고 지시하는 프롬프팅 기법
#
#   일반 프롬프트: "답을 알려줘" → 바로 결론 (틀릴 수 있음)
#   CoT 프롬프트: "단계별로 생각해봐" → 추론 과정 명시 → 정확도 ↑
#
# 수업에서 배운 예시 (260326_template):
#   - 경마 문제, 몬티홀 문제 등에서 CoT 적용 시 정답률 대폭 향상
#   - 핵심: "이 문제의 조건을 정확히 읽고 단계별로 생각하세요"
#
# 주택청약 FAQ에 CoT를 적용하는 이유:
#   "1순위인데 특별공급도 되나요?" 같은 복합 질문은
#   단순 검색만으로 답하기 어려움 → 단계별 분석이 필요!

cot_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "주택청약 전문 상담원입니다. 참고 FAQ:\n{context}\n\n"
     "복잡한 질문에는 다음 사고 과정을 따르세요:\n"
     "1단계 - 문제 파악: 사용자의 질문 핵심 파악\n"
     "2단계 - 관련 정보 정리: FAQ에서 관련 내용 추출\n"
     "3단계 - 해결 방법 제시: 단계별 안내\n\n"
     "각 단계를 명시적으로 표시하여 답변하세요."),
    ("user", "{question}")
])

cot_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | cot_prompt | llm | StrOutputParser()
)

# ── 복잡한 질문으로 CoT 테스트 ───────────────────────────────────
# 단순 FAQ 조회가 아닌, 여러 정보를 조합해야 하는 질문들
complex_qs = [
    "청약통장 1순위인데 특별공급도 신청할 수 있나요?",
    "무주택인데 부모님이 집이 있으면 청약 가능한가요?",
    "가점이 낮은데 당첨 가능성을 높이려면 어떻게 해야 하나요?"
]
for q in complex_qs:
    print(f"\n{'=' * 50}")
    print(f"❓ {q}")
    print(cot_chain.invoke(q)[:300])

---
## 사이클 4: ConversationBufferMemory

`ConversationBufferMemory`와 `MessagesPlaceholder`를 사용해 이전 대화를 기억하는 챗봇을 만드세요. 3턴 이상의 연속 대화를 테스트하세요.
`ConversationBufferMemory`가 deprecated된 `Modern Langchain`을 활용해서도 구현해보세요

In [ ]:
# ── 사이클 4: ConversationBufferMemory (Legacy) ──────────────────
# ⭐ 왜 메모리가 필요한가?
#   LLM은 기본적으로 '무상태(Stateless)' → 매 호출이 독립적
#   "청약통장이 뭐예요?" → 답변 → "1순위 조건은요?" → "뭐의 1순위요?"
#   → 이전 대화를 기억하지 못해서 맥락이 끊김!
#
# ConversationBufferMemory:
#   모든 대화를 그대로 저장하는 가장 단순한 메모리
#   장점: 구현 간단, 모든 맥락 유지
#   단점: 대화가 길어지면 토큰이 계속 늘어남 (비용 ↑)
#
# MessagesPlaceholder:
#   프롬프트 안에 "여기에 이전 대화 기록이 들어갈 자리"를 만드는 것
#   → 시스템 메시지와 사용자 메시지 사이에 대화 이력을 삽입

# 수업에서 배운 내용 (260327_template):
#   - LLM은 stateless → 이전 대화를 직접 메시지 리스트에 넣어줘야 함
#   - ConversationBufferMemory는 save_context() / load_memory_variables()로 관리

from langchain.memory import ConversationBufferMemory

# return_messages=True: 문자열 대신 Message 객체 리스트로 반환
# memory_key="history": 프롬프트의 MessagesPlaceholder 이름과 일치시킴
memory = ConversationBufferMemory(return_messages=True, memory_key="history")

# ── 메모리를 포함한 프롬프트 ─────────────────────────────────────
# MessagesPlaceholder("history") → 여기에 이전 대화가 들어감
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "주택청약 전문 상담원입니다. 참고 FAQ:\n{context}\n\n이전 대화를 참고하여 답변하세요."),
    MessagesPlaceholder(variable_name="history"),  # ← 대화 이력이 여기에 삽입됨
    ("user", "{question}")
])
memory_chain = memory_prompt | llm | StrOutputParser()

def chat_with_memory(question):
    """메모리를 사용하는 대화 함수"""
    # 1. 벡터 검색으로 관련 FAQ 가져오기
    docs = retriever.invoke(question)
    context = format_docs(docs)

    # 2. 메모리에서 이전 대화 불러오기
    history = memory.load_memory_variables({})["history"]

    # 3. context + history + question을 모두 체인에 전달
    answer = memory_chain.invoke({
        "context": context,
        "question": question,
        "history": history
    })

    # 4. 이번 대화도 메모리에 저장 (다음 턴에 사용)
    memory.save_context({"input": question}, {"output": answer})
    return answer

# ── 3턴 연속 대화 테스트 ─────────────────────────────────────────
# 3번째 질문 "수도권은 몇 개월?"은 이전 대화의 맥락(1순위)을 알아야 답할 수 있음
for q in ["청약통장이 뭐예요?", "1순위 조건은요?", "그러면 수도권은 몇 개월이에요?"]:
    print(f"\n사용자: {q}")
    print(f"챗봇: {chat_with_memory(q)[:150]}...")

print(f"\n메모리: {len(memory.load_memory_variables({})['history'])}개 메시지")

### ✅ Modern LangChain: InMemoryChatMessageHistory + RunnableWithMessageHistory

> `ConversationBufferMemory`는 LangChain v0.3+에서 **deprecated**입니다.
> `InMemoryChatMessageHistory`(저장소) + `RunnableWithMessageHistory`(자동 관리)를 사용합니다.

In [ ]:
# ── 사이클 4 (모던): InMemoryChatMessageHistory + RunnableWithMessageHistory ──
# ⭐ Modern LangChain 메모리 패턴:
#   Legacy: ConversationBufferMemory (save_context / load 수동 관리)
#   Modern: InMemoryChatMessageHistory + RunnableWithMessageHistory (자동 관리)
#
# 장점:
#   1. save_context() / load_memory_variables() 수동 호출이 필요 없음
#   2. session_id로 여러 사용자의 대화를 독립적으로 관리 가능
#   3. RunnableWithMessageHistory가 자동으로 대화를 저장/로드

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# ── 세션별 히스토리 저장소 ────────────────────────────────────────
# session_id → InMemoryChatMessageHistory 매핑
store = {}

def get_session_history(session_id: str):
    """세션 ID로 대화 기록을 가져옴 (없으면 새로 생성)"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# ── 프롬프트 + 체인 구성 ─────────────────────────────────────────
modern_prompt = ChatPromptTemplate.from_messages([
    ("system", "주택청약 전문 상담원입니다. 참고 FAQ:\n{context}\n\n이전 대화를 참고하여 답변하세요."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{question}")
])

# RunnablePassthrough.assign()으로 question에서 context를 자동 생성
modern_chain = RunnablePassthrough.assign(
    context=lambda x: format_docs(retriever.invoke(x["question"]))
) | modern_prompt | llm | StrOutputParser()

# ── RunnableWithMessageHistory로 감싸기 ──────────────────────────
# 이것만으로 대화 저장/로드가 자동화됨!
modern_chain_with_memory = RunnableWithMessageHistory(
    modern_chain,
    get_session_history,
    input_messages_key="question",    # 입력에서 사용자 메시지 키
    history_messages_key="history",   # 프롬프트에서 대화 이력 키
)

# ── 테스트 (config로 세션 지정) ──────────────────────────────────
config = {"configurable": {"session_id": "cycle4_test"}}

for q in ["청약통장이 뭐예요?", "1순위 조건은요?", "그러면 수도권은 몇 개월이에요?"]:
    answer = modern_chain_with_memory.invoke({"question": q}, config=config)
    print(f"\n사용자: {q}")
    print(f"챗봇: {answer[:150]}...")

print(f"\n메모리: {len(store['cycle4_test'].messages)}개 메시지 (자동 저장됨)")

---
## 사이클 5: ConversationBufferWindowMemory

`ConversationBufferWindowMemory(k=3)`으로 최근 3턴만 기억하는 메모리를 만들고, 5턴 대화 후 초기 대화가 잊혀지는지 확인하세요.
`ConversationBufferWindowMemory`가 deprecated된 `Modern Langchain`을 활용해서도 구현해보세요

In [ ]:
# ── 사이클 5: ConversationBufferWindowMemory (Legacy) ─────────────
# ⭐ BufferMemory vs WindowMemory:
#   Buffer: 모든 대화 저장 → 토큰 무한 증가 (비용 폭탄!)
#   Window(k=3): 최근 k턴만 저장, 오래된 건 버림 → 토큰 절약
#
# 실서비스에서는 WindowMemory가 더 실용적:
#   - 10턴 대화 후 Buffer는 수천 토큰 → Window(k=3)는 항상 3턴분
#   - 대부분의 맥락은 최근 3~5턴이면 충분
#
# k=3 의미: 최근 3턴(= 6개 메시지: user 3 + assistant 3)만 유지

from langchain.memory import ConversationBufferWindowMemory

# k=3: 최근 3턴만 메모리에 유지
window_memory = ConversationBufferWindowMemory(k=3, return_messages=True, memory_key="history")

def chat_with_window(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    history = window_memory.load_memory_variables({})["history"]
    answer = memory_chain.invoke({"context": context, "question": question, "history": history})
    window_memory.save_context({"input": question}, {"output": answer})
    return answer

# ── 5턴 대화 → 초기 대화가 잊혀지는지 확인 ───────────────────────
# 5번째 질문 "첫 번째 질문이 뭐였죠?"에서 k=3이므로
# 1~2번째 대화는 이미 잊혀져서 답하지 못할 것!
msgs = ["청약통장 가입 방법", "1순위 조건은?", "특별공급 종류는?", "가점제란?", "첫 번째 질문이 뭐였죠?"]
for q in msgs:
    print(f"\n사용자: {q}")
    print(f"챗봇: {chat_with_window(q)[:120]}...")
    h = window_memory.load_memory_variables({})["history"]
    print(f"메모리: {len(h)}개 메시지")

In [ ]:
# ── 사이클 5 (모던): 윈도우 메모리 + RunnableWithMessageHistory ────
# ⭐ Modern LangChain에는 WindowMemory가 별도로 없음
#   → InMemoryChatMessageHistory + 수동 트리밍으로 구현!
#   get_window_history() 함수에서 2K개 초과 메시지를 삭제

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

window_store = {}
K = 3  # 최근 3턴만 유지

def get_window_history(session_id: str):
    """최근 K턴만 남기는 윈도우 히스토리"""
    if session_id not in window_store:
        window_store[session_id] = InMemoryChatMessageHistory()

    history = window_store[session_id]

    # 메시지가 2K개(= K턴 × 2)를 초과하면 가장 오래된 것부터 삭제
    while len(history.messages) > K * 2:
        history.messages.pop(0)

    return history

# 위에서 만든 modern_chain 재사용
window_chain = RunnableWithMessageHistory(
    modern_chain,
    get_window_history,
    input_messages_key="question",
    history_messages_key="history",
)

# ── 테스트 ────────────────────────────────────────────────────────
config = {"configurable": {"session_id": "cycle5_test"}}

msgs = ["청약통장 가입 방법", "1순위 조건은?", "특별공급 종류는?", "가점제란?", "첫 번째 질문이 뭐였죠?"]
for q in msgs:
    answer = window_chain.invoke({"question": q}, config=config)
    h = window_store["cycle5_test"].messages
    print(f"\n사용자: {q}")
    print(f"챗봇: {answer[:120]}...")
    print(f"메모리: {len(h)}개 메시지 (최근 {len(h)//2}턴)")

---
## 사이클 6: RAG + Memory 통합 챗봇

RAG 검색 + 대화 메모리를 결합한 `FAQChatbotV3` 클래스를 만드세요. `ask(question)` 메서드와 `reset()` 메서드를 구현하고, 5턴 멀티턴 대화를 테스트하세요.

In [ ]:
# ── 사이클 6: RAG + Memory 통합 챗봇 ──────────────────────────────
# ⭐ V1(W1) → V2(W2) → V3(W3) 진화 과정:
#   V1: 키워드 검색 + LCEL 체인
#   V2: 벡터 검색(FAISS) + RAG 체인
#   V3: 벡터 검색 + RAG + 대화 메모리 + 윈도우 제한
#
# FAQChatbotV3는 지금까지 배운 모든 것을 하나의 클래스로 통합:
#   - FAISS Retriever (벡터 검색)
#   - ConversationBufferWindowMemory (최근 대화 기억)
#   - MessagesPlaceholder (프롬프트에 대화 이력 삽입)
#   - ask() / reset() 인터페이스

import time
from langchain.memory import ConversationBufferWindowMemory

class FAQChatbotV3:
    """RAG + 대화 메모리 통합 주택청약 FAQ 챗봇 (Weekend 3 최종)"""

    def __init__(self, vectorstore, llm):
        # 벡터 검색용 Retriever
        self.retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

        # 대화 메모리 (최근 5턴만 유지)
        self.memory = ConversationBufferWindowMemory(
            k=5, return_messages=True, memory_key="history"
        )

        # RAG + 메모리 프롬프트
        self.prompt = ChatPromptTemplate.from_messages([
            ("system",
             "주택청약 전문 상담원입니다.\n"
             "참고 FAQ:\n{context}\n\n"
             "이전 대화를 참고하여 답변하세요."),
            MessagesPlaceholder(variable_name="history"),
            ("user", "{question}")
        ])

        # LCEL 체인
        self.chain = self.prompt | llm | StrOutputParser()

    def ask(self, question):
        """질문에 답변합니다. answer, time, sources를 반환합니다."""
        start = time.time()

        # 1. 벡터 검색
        docs = self.retriever.invoke(question)
        context = format_docs(docs)

        # 2. 메모리에서 대화 이력 로드
        history = self.memory.load_memory_variables({})["history"]

        # 3. LLM 호출 (context + history + question)
        answer = self.chain.invoke({
            "context": context,
            "question": question,
            "history": history
        })

        # 4. 메모리에 이번 대화 저장
        self.memory.save_context({"input": question}, {"output": answer})

        return {
            "answer": answer,
            "time": round(time.time() - start, 2),
            "sources": [d.metadata for d in docs]
        }

    def reset(self):
        """대화 이력 초기화"""
        self.memory.clear()
        print("💬 대화 이력 초기화")

# ── 5턴 멀티턴 대화 테스트 ───────────────────────────────────────
chatbot = FAQChatbotV3(vectorstore, llm)
for q in [
    "청약통장이 뭐예요?",
    "1순위 조건은요?",
    "그러면 투기과열지구는?",           # 이전 맥락(1순위) 참조
    "감사합니다. 특별공급도 알려주세요",  # 화제 전환
    "신혼부부는 어떤 조건이에요?"        # 특별공급 맥락 이어감
]:
    r = chatbot.ask(q)
    print(f"\n사용자: {q}")
    print(f"챗봇: {r['answer'][:150]}... ({r['time']}초)")

---
## 사이클 7: 의도 분류기

사용자 메시지를 greeting/question/complaint/chitchat으로 분류하는 체인을 만들고, 의도별로 다른 응답 전략을 적용하세요. 5가지 메시지로 테스트하세요.

In [ ]:
# ── 사이클 7: 의도 분류기 ─────────────────────────────────────────
# ⭐ 의도 분류(Intent Classification)란?
#   사용자 메시지를 카테고리로 분류하여 적절한 처리 전략을 선택
#
#   의도 → 처리 전략:
#   greeting  (인사)     → 고정 인사 메시지 (API 호출 불필요, 비용 절약)
#   question  (청약 질문) → RAG 체인으로 답변 생성
#   complaint (불만)     → 사과 + RAG 답변
#   chitchat  (잡담)     → 청약 질문으로 안내
#
# 구현 방식:
#   LLM에게 JSON으로 의도를 분류하게 하고,
#   파싱한 결과에 따라 분기 처리 (if/elif)

import json

# ── 의도 분류 체인 ───────────────────────────────────────────────
# LLM에게 JSON만 출력하도록 강하게 지시
# {{}} 는 ChatPromptTemplate에서 중괄호 이스케이프 (리터럴 { } 출력)
intent_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "사용자 메시지의 의도를 분류하세요.\n"
     "카테고리: greeting(인사), question(청약질문), complaint(불만), chitchat(잡담)\n"
     'JSON만 출력: {{"intent": "카테고리"}}'),
    ("user", "{message}")
])
intent_chain = intent_prompt | llm | StrOutputParser()

# ── 의도별 응답 전략 ─────────────────────────────────────────────
def handle_by_intent(message):
    """의도를 분류하고, 의도에 맞는 응답을 생성합니다."""
    # 1. LLM으로 의도 분류
    raw = intent_chain.invoke({"message": message})
    raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        result = json.loads(raw)
    except:
        result = {"intent": "question"}  # 파싱 실패 시 기본값
    intent = result["intent"]

    # 2. 의도별 분기 처리
    if intent == "greeting":
        return {"intent": intent, "answer": "안녕하세요! 주택청약 상담입니다. 무엇을 도와드릴까요?"}
    elif intent == "chitchat":
        return {"intent": intent, "answer": "저는 주택청약 관련 질문에 답변드리는 챗봇입니다. 청약 관련 질문을 해주세요!"}
    elif intent == "complaint":
        r = chatbot.ask(message)
        return {"intent": intent, "answer": f"불편을 드려 죄송합니다. 도와드리겠습니다.\n\n{r['answer']}"}
    else:  # question
        r = chatbot.ask(message)
        return {"intent": intent, "answer": r["answer"]}

# ── 5가지 메시지 테스트 ──────────────────────────────────────────
for msg in [
    "안녕하세요!",                    # greeting
    "청약통장 1순위 조건이 뭐예요?",    # question
    "왜 이렇게 어려운 거야",            # complaint
    "오늘 날씨 어때?",                 # chitchat
    "특별공급 신청이 안 돼요!"          # complaint
]:
    r = handle_by_intent(msg)
    print(f"💬 '{msg}' → [{r['intent']}] {r['answer'][:80]}...")

---
## 사이클 8: 답변 불가 처리

`similarity_search_with_score`의 거리 값으로 FAQ 범위 밖 질문을 감지하세요. 답변 가능/불가를 판별하는 `smart_answer(question)` 함수를 만들고, FAQ 관련/무관 질문 각 3개로 테스트하세요.

In [ ]:
# ── 사이클 8: 답변 불가 처리 ──────────────────────────────────────
# ⭐ 답변 불가 감지(Out-of-Scope Detection)란?
#   "서울 맛집 추천해줘" 같은 FAQ 범위 밖 질문에
#   RAG가 엉뚱한 답변을 생성하는 것을 방지!
#
# 원리:
#   similarity_search_with_score()의 L2 거리 점수를 활용
#   - 점수 낮음 (< threshold) → 관련 FAQ 있음 → 답변 가능
#   - 점수 높음 (> threshold) → 관련 FAQ 없음 → 답변 불가
#
# threshold 설정:
#   FAQ 관련 질문의 점수 분포를 보고 적절한 임계값 결정
#   (보통 0.5~1.5 사이, 데이터에 따라 다름)

def smart_answer(question, threshold=10):
    """FAQ 범위 내 질문인지 판별 후 답변합니다."""
    # 1. 가장 유사한 FAQ 검색 + 점수
    results = vectorstore.similarity_search_with_score(question, k=1)

    if not results:
        return {"answer": "관련 FAQ를 찾지 못했습니다.", "answerable": False, "score": 0}

    doc, score = results[0]

    # 2. 점수가 threshold 초과 → FAQ 범위 밖
    if score > threshold:
        return {
            "answer": f"해당 질문은 주택청약 FAQ 범위를 벗어납니다.\n📞 문의: 청약홈(1644-7445)",
            "answerable": False,
            "score": score
        }

    # 3. 점수가 threshold 이하 → FAQ 범위 내 → RAG로 답변
    r = chatbot.ask(question)
    return {"answer": r["answer"], "answerable": True, "score": score}

# ── FAQ 관련/무관 질문 각 3개로 테스트 ────────────────────────────
test_qs = [
    # FAQ 관련 질문 (낮은 점수 예상)
    "청약통장 1순위 조건", "신혼부부 특별공급", "당첨 확인 방법",
    # FAQ 무관 질문 (높은 점수 예상)
    "서울 맛집 추천해줘", "오늘 주식 시장 어때?", "영화 추천해줘"
]
for q in test_qs:
    r = smart_answer(q)
    icon = "✅" if r["answerable"] else "❌"
    print(f"{icon} '{q}' (score:{r['score']:.3f}) → {r['answer'][:80]}...")

---
## 사이클 9: Gradio 최종 UI

의도 분류 + RAG + 메모리를 통합한 최종 `gr.ChatInterface`를 만드세요.

In [ ]:
# ── 사이클 9: Gradio 최종 UI ──────────────────────────────────────
# ⭐ 최종 UI는 지금까지 만든 모든 컴포넌트를 통합합니다:
#   1. 의도 분류 (handle_by_intent) → greeting/chitchat은 빠른 응답
#   2. RAG 검색 (FAQChatbotV3)     → question/complaint는 FAQ 기반 답변
#   3. 대화 메모리 (Window k=5)    → 이전 대화 맥락 유지
#
# gr.ChatInterface 콜백 함수 형태:
#   fn(message, history) → str
#   - message: 현재 사용자 입력
#   - history: Gradio가 관리하는 대화 기록 (표시용)

import gradio as gr

# 새 챗봇 인스턴스 (UI용)
chatbot_final = FAQChatbotV3(vectorstore, llm)

def final_chat(message, history):
    """의도 분류 + RAG + 메모리 통합 채팅 함수"""
    if not message or not message.strip():
        return "질문을 입력해주세요!"
    result = handle_by_intent(message)
    return result["answer"]

demo = gr.ChatInterface(
    fn=final_chat,
    title="🏠 주택청약 FAQ 챗봇 v3 (최종)",
    description="의도 분류 + RAG + 대화 메모리 통합 챗봇",
    examples=[
        "안녕하세요!",
        "청약통장이 뭔가요?",
        "1순위 조건",
        "신혼부부 특공",
        "당첨 확인 방법"
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=False, inline=True)

---
## 사이클 10: 최종 데모 & 프로젝트 회고

5턴 멀티턴 대화 데모를 실행하고, 10개 질문으로 최종 벤치마크를 돌리세요. 3주간 구현한 기능 목록과 향후 개선점 3가지를 정리하세요.

In [ ]:
# ── 사이클 10: 최종 데모 & 프로젝트 회고 ──────────────────────────
# ⭐ 최종 평가: 3주간 만든 시스템의 종합 성능 측정
import time

# ── Part 1: 멀티턴 대화 데모 ─────────────────────────────────────
# 의도 분류 + RAG + 메모리가 모두 동작하는지 확인
chatbot.reset()
print("💬 멀티턴 대화 데모:")
for q in [
    "안녕하세요!",                        # greeting → 고정 응답
    "청약통장이 뭐예요?",                  # question → RAG
    "1순위 조건은요?",                     # question → RAG + 메모리(맥락)
    "감사합니다. 특별공급도 알려주세요",     # question → 화제 전환
    "신혼부부 조건이 궁금해요"              # question → 특별공급 맥락 이어감
]:
    r = handle_by_intent(q)
    print(f"\n사용자: {q}")
    print(f"챗봇 [{r['intent']}]: {r['answer'][:150]}...")

# ── Part 2: 10개 질문 최종 벤치마크 ──────────────────────────────
print("\n\n📊 최종 벤치마크")
print("=" * 60)
chatbot.reset()

test_qs = [
    "청약통장 가입 방법", "1순위 조건", "무주택자 기준",
    "신혼부부 특별공급", "가점제 설명", "당첨 확인",
    "재당첨 제한", "청약홈 사용법", "특별공급 종류", "생애최초 특별공급",
]
total = 0
for i, q in enumerate(test_qs, 1):
    r = chatbot.ask(q)
    total += r["time"]
    cat = r["sources"][0]["category"] if r["sources"] else "N/A"
    print(f"[{i:2d}] {q} → {cat} ({r['time']}초)")

print(f"\n평균: {total / len(test_qs):.1f}초")

# ── Part 3: 3주 프로젝트 회고 ────────────────────────────────────
print("\n📋 3주 프로젝트 회고")
print("  W1: API + LCEL 체인 + 키워드 검색 + Gradio")
print("  W2: Embeddings + FAISS + RAG 체인 + 소스 표시")
print("  W3: Few-shot + CoT + Memory + 의도 분류 + 최종 UI")
print("\n🚀 향후 개선점:")
print("  1. 실제 청약 DB 연동 (HH 데이터 등)")
print("  2. 사용자 피드백 기반 답변 개선 (RLHF)")
print("  3. 음성 인터페이스 추가 (STT/TTS)")

---
## 프로젝트 1 완료! 🎉

| 주차 | 핵심 기술 | 구현 내용 |
|------|----------|----------|
| **W1** | API + LCEL | OpenAI API 직접 호출, LangChain 체인, 키워드 검색, Gradio UI |
| **W2** | Embeddings + RAG | 벡터 임베딩, FAISS, Retriever, RAG 체인, 소스 표시 |
| **W3** | Memory + Prompt | Few-shot, CoT, 대화 메모리, 의도 분류, 답변 불가 처리 |

3주간 수고하셨습니다!